# Occlusion

Entraîne une **liste de backbones** l'un après l'autre,
puis **moyenne leurs prédictions (ensemble)**, recalibre, et exporte la soumission finale.

Leviers : fine-tuning end-to-end, AMP, dégel progressif, EMA, **DIR-LDS**
(pondération par densité d'étiquettes lissée), sampler genre, soft augmentation, TTA flip,
calibration isotone, quantile mapping optionnel.



## 1. Configuration

In [1]:
import os, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from scipy.ndimage import gaussian_filter1d
import torch, torch.nn as nn
import torchvision.transforms as T, torchvision.models as models

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if device.type=="cuda" else "CPU")

# ── LES MODÈLES DE L'ENSEMBLE (entraînés l'un après l'autre, puis moyennés) ──
BACKBONES = ["convnext_small", "convnext_base"]   # 1 seul = plus simple/rapide ; ajoute "swin_s" si le temps le permet

IMG_SIZE=224
BATCH={"convnext_tiny":64,"convnext_small":48,"convnext_base":32,"swin_s":32,"efficientnet_b3":48}
EPOCHS=22
FREEZE_EPOCHS=2
LR_HEAD=1e-3; LR_BACKBONE=4e-5; WEIGHT_DECAY=5e-2
EMA_DECAY=0.999; GRAD_CLIP=1.0
VAL_FRAC=0.20; NUM_WORKERS=4; SEED=42

GENDER_RATIO=1.5
LDS_BINS=100; LDS_SIGMA=2.0; OMEGA_CLIP=10.0
FAIRNESS_LAMBDA=0.0          # 0=off ; essaie 0.5-2.0 si Err_M/Err_F se déséquilibrent
USE_QUANTILE_MAP=False       # expérimental (voir cellule finale)

OUT_DIR="/kaggle/working"; MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
torch.manual_seed(SEED); np.random.seed(SEED)
print("Ensemble:", BACKBONES)


Device: cuda | Tesla T4
Ensemble: ['convnext_small', 'convnext_base']


## 2. Chemins

In [2]:
INPUT="/kaggle/input"; base=INPUT.rstrip("/").count("/")
TRAIN_CSV=TEST_CSV=None
for dp,dns,fns in os.walk(INPUT):
    if dp.count("/")-base>=5: dns[:]=[]
    if TRAIN_CSV is None and "train.csv" in fns: TRAIN_CSV=os.path.join(dp,"train.csv")
    if TEST_CSV  is None and "test_students.csv" in fns: TEST_CSV=os.path.join(dp,"test_students.csv")
assert TRAIN_CSV and TEST_CSV,"CSV introuvables"
sample_rel=pd.read_csv(TRAIN_CSV,nrows=1)["filename"].iloc[0]
IMAGE_DIR=None
for dp,dns,_ in os.walk(INPUT):
    if dp.count("/")-base>=5: dns[:]=[]
    if os.path.exists(os.path.join(dp,sample_rel)): IMAGE_DIR=dp; break
assert IMAGE_DIR,"Images introuvables (dossier crops/ ?)"
print("IMAGE_DIR:",IMAGE_DIR)


IMAGE_DIR: /kaggle/input/datasets/mouhamedsamb2001/data-challenge-dataset/crops/Crop_224_5fp_100K


## 3. Données, split, poids DIR-LDS, loaders

In [3]:
from sklearn.model_selection import train_test_split
df_all=pd.read_csv(TRAIN_CSV).dropna().reset_index(drop=True)
df_test=pd.read_csv(TEST_CSV).dropna().reset_index(drop=True)
df_train,df_val=train_test_split(df_all,test_size=VAL_FRAC,stratify=df_all["gender"],random_state=SEED)
df_train=df_train.reset_index(drop=True); df_val=df_val.reset_index(drop=True)
yv,gv=df_val["FaceOcclusion"].values, df_val["gender"].values
print("Train",len(df_train),"Val",len(df_val))

def lds_weights(gt,n=LDS_BINS,sigma=LDS_SIGMA,clip=OMEGA_CLIP):
    edges=np.linspace(0,1,n+1); idx=np.clip(np.digitize(gt,edges)-1,0,n-1)
    hist=np.bincount(idx,minlength=n).astype(float)
    eff=np.maximum(gaussian_filter1d(hist,sigma),1e-6); p=eff[idx]/eff.sum()
    om=(1/30+gt)/p; om/=om.mean(); return np.clip(om,0,clip).astype(np.float32)
omega_train=lds_weights(df_train["FaceOcclusion"].values)
print(f"Ω min {omega_train.min():.2f} max {omega_train.max():.2f} mean {omega_train.mean():.2f}")

transform_train=T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)),T.RandomHorizontalFlip(),
    T.ColorJitter(0.2,0.2,0.15),T.RandomApply([T.GaussianBlur(3,(0.1,1.5))],p=0.2),
    T.ToTensor(),T.Normalize(MEAN,STD)])
transform_eval=T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)),T.ToTensor(),T.Normalize(MEAN,STD)])
transform_flip=T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)),T.RandomHorizontalFlip(p=1.0),T.ToTensor(),T.Normalize(MEAN,STD)])

class DS(torch.utils.data.Dataset):
    def __init__(self,df,tf,train=True,om=None):
        self.df=df.reset_index(drop=True); self.tf=tf; self.train=train; self.om=om
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.loc[i]; img=Image.open(os.path.join(IMAGE_DIR,r["filename"])).convert("RGB"); x=self.tf(img)
        if self.train: return x,np.float32(r["FaceOcclusion"]),np.float32(r["gender"]),np.float32(self.om[i])
        return x,r["filename"]

g=df_train["gender"].values
sampler=torch.utils.data.WeightedRandomSampler(np.where(g==0.0,GENDER_RATIO,1.0).astype(np.float64),len(g),replacement=True)
def make_train_loader(bs):
    return torch.utils.data.DataLoader(DS(df_train,transform_train,True,omega_train),batch_size=bs,
        sampler=sampler,num_workers=NUM_WORKERS,pin_memory=True,drop_last=True)
val_loader=torch.utils.data.DataLoader(DS(df_val,transform_eval,True,np.ones(len(df_val),np.float32)),
    batch_size=64,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)
test_loader=torch.utils.data.DataLoader(DS(df_test,transform_eval,False),batch_size=64,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)
test_loader_flip=torch.utils.data.DataLoader(DS(df_test,transform_flip,False),batch_size=64,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)


Train 80000 Val 20000
Ω min 0.01 max 10.00 mean 0.35


## 4. Modèle, loss, métrique, EMA

In [4]:
def build_backbone(name):
    if name=="convnext_tiny": bb=models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1); bb.classifier[2]=nn.Identity(); f=768
    elif name=="convnext_small": bb=models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1); bb.classifier[2]=nn.Identity(); f=768
    elif name=="convnext_base": bb=models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1); bb.classifier[2]=nn.Identity(); f=1024
    elif name=="swin_s": bb=models.swin_s(weights=models.Swin_S_Weights.IMAGENET1K_V1); bb.head=nn.Identity(); f=768
    elif name=="efficientnet_b3": bb=models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1); bb.classifier=nn.Identity(); f=1536
    else: raise ValueError(name)
    return bb,f
class Net(nn.Module):
    def __init__(self,name):
        super().__init__(); self.backbone,f=build_backbone(name)
        self.head=nn.Sequential(nn.Linear(f,256),nn.GELU(),nn.Dropout(0.3),nn.Linear(256,1),nn.Sigmoid())
    def forward(self,x): return self.head(self.backbone(x))

def omega_mse(p,gt,om): return (om*(p.squeeze(-1)-gt)**2).mean()
def fairness_term(p,gt,gd,om):
    pr=p.squeeze(-1); se=om*(pr-gt)**2; m,f=gd==1.0,gd==0.0
    if m.sum()==0 or f.sum()==0: return torch.zeros((),device=p.device)
    em=se[m].sum()/(om[m].sum()+1e-8); ef=se[f].sum()/(om[f].sum()+1e-8)
    return torch.sqrt((em-ef)**2+1e-8)
def error_fn(p,gt): w=1/30+gt; return float(np.sum(w*(p-gt)**2)/np.sum(w))
def metric_fn(p,gt,gd):
    m,f=gd==1.0,gd==0.0; em,ef=error_fn(p[m],gt[m]),error_fn(p[f],gt[f]); return (em+ef)/2+abs(em-ef),em,ef
class EMA:
    def __init__(self,model,decay): self.decay=decay; self.shadow={k:v.detach().clone() for k,v in model.state_dict().items()}
    @torch.no_grad()
    def update(self,model):
        for k,v in model.state_dict().items():
            if v.dtype.is_floating_point: self.shadow[k].mul_(self.decay).add_(v.detach(),alpha=1-self.decay)
            else: self.shadow[k]=v.detach().clone()


## 5. Fonction d'entraînement d'un modèle

In [5]:
def infer(model,loader):
    model.eval(); out=[]
    with torch.inference_mode():
        for b in loader:
            with torch.autocast("cuda",dtype=torch.float16): out.append(model(b[0].to(device)).squeeze(-1).float().cpu().numpy())
    return np.concatenate(out)

def train_model(name):
    print(f"\n========== {name} ==========")
    bs=BATCH.get(name,32); train_loader=make_train_loader(bs)
    model=Net(name).to(device)
    scaler=torch.cuda.amp.GradScaler()
    opt=torch.optim.AdamW([{"params":model.backbone.parameters(),"lr":LR_BACKBONE},
                           {"params":model.head.parameters(),"lr":LR_HEAD}],weight_decay=WEIGHT_DECAY)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS,eta_min=1e-6)
    ema=EMA(model,EMA_DECAY)
    def eval_ema():
        bak={k:v.detach().clone() for k,v in model.state_dict().items()}
        model.load_state_dict(ema.shadow); pv=infer(model,val_loader); model.load_state_dict(bak); return pv
    best=float("inf"); best_sd=None
    for ep in range(1,EPOCHS+1):
        tb=ep>FREEZE_EPOCHS
        for p in model.backbone.parameters(): p.requires_grad=tb
        model.train()
        if not tb: model.backbone.eval()
        losses=[]
        for X,y,gd,om in tqdm(train_loader,desc=f"{name} ep{ep}/{EPOCHS}[{'e2e' if tb else 'head'}]"):
            X,y,gd,om=X.to(device),y.to(device),gd.to(device),om.to(device)
            opt.zero_grad()
            with torch.autocast("cuda",dtype=torch.float16):
                pred=model(X); loss=omega_mse(pred,y,om)
                if FAIRNESS_LAMBDA>0: loss=loss+FAIRNESS_LAMBDA*fairness_term(pred,y,gd,om)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
            scaler.step(opt); scaler.update(); ema.update(model); losses.append(loss.item())
        sched.step()
        pv=eval_ema(); sc,em,ef=metric_fn(pv,yv,gv)
        print(f"  ep{ep} loss={np.mean(losses):.5f} Score={sc:.5f} Err_M={em:.5f} Err_F={ef:.5f}")
        if sc<best: best=sc; best_sd={k:v.clone() for k,v in ema.shadow.items()}
    # meilleur EMA → prédictions val + test (TTA)
    model.load_state_dict(best_sd)
    val_pred=infer(model,val_loader)
    test_pred=(infer(model,test_loader)+infer(model,test_loader_flip))/2.0
    torch.save(best_sd,os.path.join(OUT_DIR,f"best_{name}.pt"))
    np.save(os.path.join(OUT_DIR,f"val_pred_{name}.npy"),val_pred)
    np.save(os.path.join(OUT_DIR,f"test_pred_{name}.npy"),test_pred)
    print(f"  >>> {name} meilleur Score val = {best:.5f}")
    del model; torch.cuda.empty_cache()
    return val_pred,test_pred,best


## 6. Entraînement de tous les modèles de l'ensemble

In [6]:
val_preds, test_preds = {}, {}
for name in BACKBONES:
    vp,tp,_=train_model(name)
    val_preds[name]=vp; test_preds[name]=tp
print("\nModèles entraînés:", list(val_preds))



========== convnext_small ==========
Downloading: "https://download.pytorch.org/models/convnext_small-0c510722.pth" to /root/.cache/torch/hub/checkpoints/convnext_small-0c510722.pth


100%|██████████| 192M/192M [00:00<00:00, 206MB/s]
/tmp/ipykernel_23/1011015829.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler()


convnext_small ep1/22[head]:   0%|          | 0/1666 [00:00<?, ?it/s]

  ep1 loss=0.00232 Score=0.01617 Err_M=0.01361 Err_F=0.00849


convnext_small ep2/22[head]:   0%|          | 0/1666 [00:00<?, ?it/s]

...

  ep20 loss=0.00014 Score=0.00149 Err_M=0.00132 Err_F=0.00097


convnext_base ep21/22[e2e]:   0%|          | 0/2500 [00:00<?, ?it/s]

  ep21 loss=0.00009 Score=0.00150 Err_M=0.00133 Err_F=0.00100


convnext_base ep22/22[e2e]:   0%|          | 0/2500 [00:00<?, ?it/s]

  ep22 loss=0.00012 Score=0.00151 Err_M=0.00134 Err_F=0.00099
  >>> convnext_base meilleur Score val = 0.00149

Modèles entraînés: ['convnext_small', 'convnext_base']


## 7. Ensemble + calibration + (option) quantile mapping + export

In [7]:
np.save(os.path.join(OUT_DIR,"y_val.npy"),yv); np.save(os.path.join(OUT_DIR,"g_val.npy"),gv)

# scores individuels
for n in BACKBONES:
    sc,_,_=metric_fn(val_preds[n],yv,gv); print(f"  {n:16s} Score val = {sc:.5f}")

# moyenne d'ensemble
val_ens=np.mean([val_preds[n] for n in BACKBONES],axis=0)
test_ens=np.mean([test_preds[n] for n in BACKBONES],axis=0)
sc,em,ef=metric_fn(val_ens,yv,gv); print(f"\nEnsemble (brut) Score={sc:.5f} Err_M={em:.5f} Err_F={ef:.5f}")

# calibration isotone
from sklearn.isotonic import IsotonicRegression
iso=IsotonicRegression(out_of_bounds="clip",y_min=0.0,y_max=1.0); iso.fit(val_ens,yv,sample_weight=1/30+yv)
sc_c,_,_=metric_fn(iso.predict(val_ens),yv,gv); USE_CAL=sc_c<sc
print(f"Ensemble (isotone) Score={sc_c:.5f} | appliquée:{USE_CAL}")
print(f">>> Meilleur Score ensemble val = {min(sc,sc_c):.5f}")

final=iso.predict(test_ens) if USE_CAL else test_ens

if USE_QUANTILE_MAP:
    TARGET_HIST=[(0.02,1.4),(0.06,1.0),(0.10,1.0),(0.14,0.95),(0.18,1.0),(0.22,0.95),
                 (0.26,0.9),(0.30,0.8),(0.34,0.6),(0.38,0.45),(0.42,0.3),(0.46,0.15),(0.50,0.05)]
    c=np.array([a for a,_ in TARGET_HIST]); fq=np.array([b for _,b in TARGET_HIST]); fq/=fq.sum()
    tgt=np.clip(np.random.choice(c,200000,p=fq)+np.random.uniform(-0.02,0.02,200000),0,1)
    tk=np.linspace(0,1,1001); final=np.interp(final,np.quantile(final,tk),np.quantile(tgt,tk))

final=np.clip(final,0.0,1.0)
res=pd.DataFrame({"filename":df_test["filename"].values,"FaceOcclusion":final}); res["gender"]="x"
out=os.path.join(OUT_DIR,"test_predictions.csv"); res.to_csv(out,index=False)
print("\nExport →",out,"| mean",round(final.mean(),4))
res.head()


  convnext_small   Score val = 0.00137
  convnext_base    Score val = 0.00149

Ensemble (brut) Score=0.00140 Err_M=0.00131 Err_F=0.00112
Ensemble (isotone) Score=0.00137 | appliquée:True
>>> Meilleur Score ensemble val = 0.00137

Export → /kaggle/working/test_predictions.csv | mean 0.1667


,filename,FaceOcclusion,gender
0,database3/database3/m.0256vn/81-FaceId-0_align...,0.043302,x
1,database3/database3/m.01dgwg/29-FaceId-0_align...,0.025493,x
2,database3/database3/m.02l0nk/72-FaceId-0_align...,0.025493,x
3,database3/database3/m.026jp3/94-FaceId-1_align...,0.052110,x
4,database3/database3/m.01ncgyv/24-FaceId-0_alig...,0.032737,x
